# 🎯 THỬ THÁCH TRANG GIẤY TRẮNG #1

 Nhiệm vụ của trò:

 Hãy viết một script Python hoàn chỉnh thực hiện các việc sau:

 - Dữ liệu: Tạo ra 2 câu tiếng Anh ngắn: "Hello world" và "AI is great".

- Từ điển: Tạo một vocab (từ điển) giả định để mã hóa từng từ trong 2 câu đó thành ID (số nguyên).

 - Embeddings: Sử dụng torch.nn.Embedding của PyTorch để chuyển các ID đó thành vector (mỗi từ = vector 4 chiều).

 - Kết quả: In ra shape của tensor kết quả.

In [ ]:
import torch
import torch.nn as nn
import math

sentences = ["Hello world", "AI is great"] 
vocab = {} # {word : idx}
idx = 0
for sentence in sentences:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = idx
            idx += 1

all_ids = []
for sentence in sentences:
    for word in sentence.split():
        all_ids.append(vocab[word])
input_ids = torch.tensor(all_ids)

seq_len = len(input_ids) # 5 từ
d_model = 4

# tạo ra 1 lớp nhúng và nhúng input_ids vào
token_embedder = nn.Embedding(num_embeddings=10, embedding_dim=d_model)
word_vectors = token_embedder(input_ids)
print(word_vectors)



tensor([0, 1, 2, 3, 4])
tensor([[-1.2755,  0.2457,  1.1780, -1.0662],
        [ 0.3944,  0.1363,  1.1438, -0.8922],
        [ 0.6239,  0.7012, -0.9634,  1.5893],
        [-0.3747, -0.7583, -0.1772,  1.7938],
        [-0.7128, -0.1942,  0.5546, -0.4465]], grad_fn=<EmbeddingBackward0>)


# 🎯 THỬ THÁCH TRANG GIẤY TRẮNG #2
Lần này, trò KHÔNG được phép dùng hàm torch.tril().
Nhiệm vụ của trò là: Dùng vòng lặp for của Python để tự tay xây dựng ra ma trận mặt nạ này dưới dạng một List 2 chiều (List of Lists), sau đó biến nó thành Tensor.

In [ ]:
import torch

seq_len = 4 # Giả sử câu có 4 từ
mask_list = [] # Chứa kết quả mặt nạ

# ==========================================
# VIẾT CODE CỦA TRÒ Ở ĐÂY
# Gợi ý: Dùng 2 vòng lặp for (for hang in... và for cot in...)
# ==========================================
for rows in range(seq_len):
    temp_arr = []
    for cols in range(seq_len):
        if rows >= cols:
            temp_arr.append(1)
        else:
            temp_arr.append(0)

    mask_list.append(temp_arr)


# ==========================================

# Biến List của trò thành PyTorch Tensor
mask_tensor = torch.tensor(mask_list)

print("Mặt nạ trò tự chế tạo:")
print(mask_tensor)

Mặt nạ trò tự chế tạo:
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]])


# 🎯 THỬ THÁCH TRANG GIẤY TRẮNG #3: 

Nhiệm vụ của trò: 

- Không nhìn lại code cũ, mở một trang giấy trắng trong IDE và tự code kịch bản sau:

- Khởi tạo dữ liệu: Tạo một ma trận x ngẫu nhiên đại diện cho 1 câu nói dài 5 từ, mỗi từ có vector 8 chiều (d_model = 8).

- Khởi tạo trạm kiểm soát: Gọi các hàm nn.LayerNorm, nn.MultiheadAttention, nn.Linear, nn.ReLU.

- Vòng lặp Vĩ đại: Tạo một vòng lặp for chạy 6 lần (Đại diện cho 6 lớp Transformer Block giống kiến trúc gốc).

- Bên trong vòng lặp: Cho x tự chảy qua chu trình chuẩn:

- Lưu x gốc -> Chuẩn hóa -> Đi qua Attention -> Cộng phần dư

- Lưu x mới -> Chuẩn hóa -> Đi qua Feed-Forward (Linear -> ReLU -> Linear) -> Cộng phần dư

- Sau khi kết thúc vòng lặp 6 lần, in ra shape cuối cùng của x để chứng minh rằng dù đi qua bao nhiêu tầng sâu, kích thước dữ liệu không bao giờ bị vỡ.

In [ ]:
import torch
import torch.nn as nn

d_model = 8
x = torch.randn(1, 5, d_model)

# khởi tạo lớp chuẩn hóa
ln1 = nn.LayerNorm(d_model)
ln2 = nn.LayerNorm(d_model)

# khởi tạo lớp attention
# cú pháp nn.MultiheadAttention và truyền vào 3 đối số:

# 1. embed_dim: Kích thước (số chiều) của 1 từ.

# 2. num_heads: Số lượng "cái đầu" (nhóm phân tích) cùng đọc câu văn này.

# 3. batch_first: Quy định vị trí của chiều "Batch" trong Tensor đầu vào. 

attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, batch_first=True) 

# Khởi tạo Mạng Suy nghĩ sâu (Feed-Forward Network)
# nn.Sequential giống như một cái ống nước, dữ liệu chảy qua từng lớp từ trên xuống dưới.
ffn = nn.Sequential(

    # 1. BƯỚC PHÓNG TO (Expand):
    # in_features: nhận vào vector kích thước gốc (ví dụ: d_model = 8)
    # out_features: phóng to vector lên gấp 4 lần (thành 32 chiều)
    nn.Linear(in_features=d_model, out_features=d_model * 4),
    
    # 2. BƯỚC KÍCH HOẠT (Activate):
    # Tạo "nếp nhăn" cho bộ não. Bẻ gãy các đường thẳng thành đường cong.
    # Lọc bỏ các thông tin rác (âm) thành 0, chỉ giữ lại tín hiệu mạnh.
    nn.ReLU(),
    
    # 3. BƯỚC THU NHỎ (Project back):
    # Nhận lại vector 32 chiều và nén nó về lại kích thước ban đầu (8 chiều)
    nn.Linear(in_features=d_model * 4, out_features=d_model)
)

for i in range(6):
    # bước 1: lưu x gốc
    x_goc_1 = x

    # bước 2: chuẩn hóa
    x_norm_1 = ln1(x)

    # bước 3: đi qua attention
    # attention cần truyền vào 3 đối số là Q K V tương ứng với x sau khi chuẩn hóa ở trên 
    # kết quả của biến x_attention sẽ là 1 list gồm giá trị, trọng số nên ta phải truy cập phần tử đầu tiên để lấy giá trị ra
    x_attention = attention(x_norm_1, x_norm_1, x_norm_1)[0]

    # bước 4: cộng phần dư
    x = x_goc_1 + x_attention 

    # bước 5: lưu x gốc
    x_goc_2 = x

    # bước 6: chuẩn hóa
    x_norm_2 = ln2(x)

    # bước 7: đi qua ffn
    x_ffn = ffn(x_norm_2)

    # bước 8: cộng phần dư
    x = x_goc_2 + x_ffn

    print(f"kích thước của mảng sau {i+1} vòng lặp",x.shape)






kích thước của mảng sau 1 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 2 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 3 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 4 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 5 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 6 vòng lặp torch.Size([1, 5, 8])

kích thước của mảng sau 1 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 2 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 3 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 4 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 5 vòng lặp torch.Size([1, 5, 8])
kích thước của mảng sau 6 vòng lặp torch.Size([1, 5, 8])
